# Tutorial 4: MPS simulator backends (cuQuantum, quimb, Qiskit Aer)

For molecules whose CI vector is too large for `ffsim`, SQD-GQE simulates its circuits as a **matrix product state (MPS)**. This notebook runs the same circuit through three MPS simulators, draws shots from each, and scores every set of shots with the same SQD diagonalization:

- **cuQuantum (cuTensorNet)** on an NVIDIA GPU. This is the production sampler, `sampler.kind=mps` / `mps_topk`.
- **quimb**'s `CircuitMPS`, on the CPU or, with CuPy, on the GPU.
- **Qiskit Aer**'s `matrix_product_state` method, on the CPU.

The notebook has two parts. **Part I** uses cuQuantum through sqd-gqe, the way training and inference do: the sampler comes from the config. **Part II** calls each simulator directly, without sqd-gqe's samplers: cuQuantum's `NetworkState` API (what the sqd-gqe sampler does inside), and quimb and Aer through [qiskit-addon-aqc-tensor](https://qiskit.github.io/qiskit-addon-aqc-tensor/). Only cuQuantum is integrated into sqd-gqe. We use Sn(OH)3Me in its 16-orbital, 24-electron active space (`molecule=snoh3_s`, 32 qubits). At this size the bond dimension cap truncates the MPS and the GPU pays off, and an exact `ffsim` reference still exists. The notebook takes about 30 minutes on one GB200, most of it in Aer. Set `MOLECULE = "n2"` in section 1 for a 16-qubit run that takes about two minutes.

Install, on top of `pip install -e .`:

```bash
pip install cuquantum-python-cu13 cupy-cuda13x                    # cuQuantum, NVIDIA GPU only
pip install 'qiskit-addon-aqc-tensor[aer,quimb]' 'quimb<1.12'     # quimb 1.12+ needs scipy>=1.16
```

Each backend's cell skips itself when its packages or a GPU are missing. On ROQUO, `slurm/mps_runtime_env.sh` installs cuQuantum and CuPy for you. How the cuQuantum defaults were chosen, with measurements at 28-38 qubits, is in `docs/mps.md`.

## 1. The system, the pool and one circuit
The circuit is built once, from the pool, and transpiled with the production `MpsSampler` into one- and two-qubit gates. Every backend below gets exactly this transpiled circuit.

In [1]:
import os, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "tutorial" else Path.cwd()
sys.path.insert(0, str(ROOT))      # this checkout, in case the package is installed from another
os.chdir(ROOT)                     # molecule configs give data paths relative to the repo root

from hydra import compose, initialize_config_dir
from sqd_gqe.factory import SqdFactory
from sqd_gqe.gqe.mps_sampler import MpsSampler

MOLECULE, SHOTS, CHI = "snoh3_s", 20_000, 256    # "n2" runs the same notebook at 16 qubits
with initialize_config_dir(version_base="1.3", config_dir=str(ROOT / "configs_sqd")):
    cfg = compose(config_name="default", overrides=[
        f"molecule={MOLECULE}", "reference_keys=[]", f"sampler.shots={SHOTS}",
        f"sampler.mps_max_bond={CHI}", "sqd.diag_backend=sqd", "sqd.recover_iterations=0"])

factory = SqdFactory()
system = factory.build_system(cfg)
pool = factory.build_operator_pool(cfg)
norb, nelec = system.norb, tuple(system.nelec)
print(f"{MOLECULE}: {2 * norb} qubits, nelec={nelec}, RHF {system.hf.e_tot:.6f} Ha")

# one LUCJ block among local Givens rotations, the kind of circuit the policy emits
names = [op.name for op in pool]
lucj = [i for i, n in enumerate(names) if n.startswith("LUCJ")]
givens = [i for i, n in enumerate(names) if n.startswith("givens")]
seq = [0, givens[0], givens[1], lucj[0], givens[2], givens[3]]
print("tokens:", [names[i] for i in seq])

prep = MpsSampler(pool, shots_count=1, verbose=False)        # builds and transpiles, no GPU
tqc, _ = prep.transpile(prep.prepare_circuit(seq))
print(f"transpiled: {tqc.size()} gates, {tqc.num_qubits} qubits")

converged SCF energy = -479.121325315409


Parsing molecules/small/SnOH3-me_S_16o24e_mp2no_def2svp.fcidump


snoh3_s: 32 qubits, nelec=(12, 12), RHF -479.121325 Ha
tokens: ['I', 'givens(0,1;0.050)', 'givens(0,1;0.151)', 'LUCJ(reps=2)', 'givens(1,2;0.050)', 'givens(1,2;0.151)']


transpiled: 8492 gates, 32 qubits


## 2. Shared helpers
SQD takes counts keyed by `(alpha_occupation, beta_occupation)` integers, alpha on qubits `[0, norb)` and beta on `[norb, 2*norb)`. The backends return bitstrings in different orders: quimb puts qubit 0 first, Aer (like Qiskit) puts it last. One converter handles both, and one pipeline scores all of them.

Each backend gets two times. **Build** is making the state alone. **End-to-end** runs from the circuit to the counts: build, any copy between devices, sampling, and decoding. Sampling cost differs a lot between backends, so the build time alone can mislead. All runs follow a warm-up, so CUDA start-up is not counted.

In [2]:
from collections import Counter

def to_counts(bitstring_counts, qubit0_first):
    # {bitstring: n} -> {(alpha, beta): n}
    mask = (1 << norb) - 1
    out = Counter()
    for bits, n in bitstring_counts.items():
        b = bits if qubit0_first else bits[::-1]          # now b[i] is qubit i
        s = int(b[::-1], 2)                                # integer with bit i = qubit i
        out[(s & mask, s >> norb)] += int(n)
    return dict(out)

def valid_fraction(counts):
    total = sum(counts.values())
    ok = sum(n for (a, b), n in counts.items()
             if a.bit_count() == nelec[0] and b.bit_count() == nelec[1])
    return ok / total if total else 0.0

pipeline = factory.build_pipeline(cfg)                     # SQD scoring only, sampler unused
results = {}

def fmt(seconds):
    return "-" if seconds is None else f"{seconds:.2f}"

def score(name, counts, build, e2e):
    # build: the state alone. e2e: circuit in, counts out (None when not separable)
    energy = pipeline.energy_of_counts(counts)
    results[name] = dict(energy=energy, build=build, e2e=e2e, distinct=len(counts),
                         valid=valid_fraction(counts))
    print(f"{name:22s} build {fmt(build):>6s} s  end-to-end {fmt(e2e):>6s} s  "
          f"distinct={len(counts):6d}  valid={results[name]['valid']:.3f}  E={energy:.6f} Ha")

[factory] recover_subspace_dim=45 (derived from max_dim=2000, net CI dim target ~2025)


## 3. Exact reference: ffsim
`ffsim` holds the exact state in the particle-number sector, so every shot is valid and there is no truncation. The MPS backends should reproduce this energy up to shot noise, and shot noise is large here: one rare determinant more or less in the sampled set moves the SQD energy by several mHa. So we draw the exact shots with five seeds and use their mean and spread as the yardstick. `sampler.kind=topk` reads the top-K determinants off the same exact state, under the same `1/shots` floor as `mps_topk`. It is the reference for `mps_topk`, which draws no shots.

In [3]:
from sqd_gqe.gqe.sampler import FfsimSampler

exact = FfsimSampler(pool, shots_count=SHOTS, seed=0)
t = time.perf_counter()
exact.prepare_state(seq)
build = time.perf_counter() - t
t = time.perf_counter()
counts = exact.sample_state(seq, seed=1)                    # prepares the state again
score("ffsim (exact)", counts, build, time.perf_counter() - t)
spread = [pipeline.energy_of_counts(exact.sample_state(seq, seed=s)) for s in range(1, 6)]
print(f"exact over 5 seeds: {min(spread):.6f} to {max(spread):.6f} Ha "
      f"(spread {1e3 * (max(spread) - min(spread)):.1f} mHa)")

topk_ref_cfg = cfg.copy()
topk_ref_cfg.sampler.kind = "topk"
exact_topk = factory.build_sampler(topk_ref_cfg)
t = time.perf_counter()
score("ffsim topk (exact)", exact_topk.sample_state(seq), build, time.perf_counter() - t)

ffsim (exact)          build   3.04 s  end-to-end   3.30 s  distinct=    45  valid=1.000  E=-479.156595 Ha


exact over 5 seeds: -479.164861 to -479.152392 Ha (spread 12.5 mHa)


ffsim topk (exact)     build   3.04 s  end-to-end   3.26 s  distinct=    18  valid=1.000  E=-479.114830 Ha


# Part I: cuQuantum through sqd-gqe

## 4. `sampler.kind=mps` and `mps_topk`
`sampler.kind=mps` builds `MpsSampler` from the config: cuQuantum's `NetworkState` with an `MPSConfig` (bond dimension `sampler.mps_max_bond`, cutoffs `1e-5`, `complex64`, and the `simple`-update gauge, which keeps truncation accurate at a fixed bond dimension). It detects cuQuantum's bitstring order with a Hartree-Fock probe, so its counts come back already decoded.

`sampler.kind=mps_topk` reads the most probable determinants off the same MPS instead of drawing shots. It keeps only determinants that a run of `sampler.shots` would be expected to draw at least once (`sampler.min_prob`, default `1/shots`).

End-to-end here is one `sample_state` call, including the sampler's own transpile. `mps` builds and samples in one cuQuantum call, so it has no separate build time. `mps_topk` records its build in `last_timings["build"]`.

In [4]:
try:
    import cupy, cuquantum                                   # noqa: F401
    HAS_GPU = cupy.cuda.runtime.getDeviceCount() > 0
except Exception as exc:                                     # no cuQuantum, CuPy or GPU
    HAS_GPU = False
    print("cuQuantum skipped:", type(exc).__name__, exc)

if HAS_GPU:
    mps_cfg = cfg.copy()
    mps_cfg.sampler.kind = "mps"
    mps = factory.build_sampler(mps_cfg)
    mps.verbose = False
    mps.sample_state(seq, seed=1)                            # warm-up: CUDA init and bit-order probe
    t = time.perf_counter()
    counts = mps.sample_state(seq, seed=1)
    score("sqd-gqe mps", counts, None, time.perf_counter() - t)

    topk_cfg = cfg.copy()
    topk_cfg.sampler.kind = "mps_topk"
    topk = factory.build_sampler(topk_cfg)
    topk.verbose = False
    topk.sample_state(seq)                                   # warm-up
    t = time.perf_counter()
    counts = topk.sample_state(seq)
    score("sqd-gqe mps_topk", counts, topk.last_timings["build"], time.perf_counter() - t)
    print(f"mps_topk floor: |c|^2 >= {topk.min_prob:g}, captured weight {topk.last_captured_weight:.4f}")

sqd-gqe mps            build      - s  end-to-end   5.62 s  distinct=    55  valid=1.000  E=-479.162448 Ha


sqd-gqe mps_topk       build   4.46 s  end-to-end   5.19 s  distinct=    18  valid=1.000  E=-479.114830 Ha
mps_topk floor: |c|^2 >= 5e-05, captured weight 1.0000


# Part II: calling the simulators directly

## 5. cuQuantum's `NetworkState`
This is what `MpsSampler` does inside, written out. `NetworkState.from_circuit` builds the MPS on the GPU under an `MPSConfig`, and `compute_sampling` draws shots. The settings below are sqd-gqe's defaults.

Build is `from_circuit` plus `compute_output_state`, which runs the contraction and returns the MPS tensors. End-to-end is a fresh state with `compute_sampling`, which contracts the same MPS and draws all shots in one batch on the GPU.

cuQuantum's bitstring order is not documented as Qiskit's, so check it before decoding: sample a gate-free Hartree-Fock circuit and see which end the occupied qubits come out at. (`MpsSampler` runs the same probe once per sampler. cuQuantum 26.x returned qubit 0 first.)

In [5]:
if HAS_GPU:
    from cuquantum.tensornet.experimental import MPSConfig, NetworkState

    config = MPSConfig(max_extent=CHI, abs_cutoff=1e-5, rel_cutoff=1e-5, gauge_option="simple")

    def cuquantum_state(circuit):
        return NetworkState.from_circuit(circuit, dtype="complex64", config=config, backend="cupy")

    def cuquantum_sample(circuit, shots, seed):
        state = cuquantum_state(circuit)
        try:
            return state.compute_sampling(shots, seed=seed)       # {bitstring: count}
        finally:
            state.free()

    def cuquantum_build(circuit):
        state = cuquantum_state(circuit)
        try:
            tensors = state.compute_output_state()                # the MPS site tensors
            cupy.cuda.Device().synchronize()
            return tensors
        finally:
            state.free()

    hf_circuit, _ = prep.transpile(prep.prepare_circuit([]))
    hf_bits = next(iter(cuquantum_sample(hf_circuit, 10, seed=1)))
    occupied = "".join("1" if q < nelec[0] or norb <= q < norb + nelec[1] else "0"
                       for q in range(2 * norb))
    qubit0_first = hf_bits == occupied
    assert qubit0_first or hf_bits[::-1] == occupied, hf_bits
    print("Hartree-Fock sample:", hf_bits, "-> qubit 0 first" if qubit0_first else "-> qubit 0 last")

    cuquantum_build(tqc)                                      # warm-up
    cuquantum_sample(tqc, SHOTS, seed=1)
    t = time.perf_counter()
    cuquantum_build(tqc)
    build = time.perf_counter() - t
    t = time.perf_counter()
    counts = to_counts(cuquantum_sample(tqc, SHOTS, seed=1), qubit0_first)
    score("cuQuantum (direct)", counts, build, time.perf_counter() - t)

Hartree-Fock sample: 11111111111100001111111111110000 -> qubit 0 first


cuQuantum (direct)     build   5.09 s  end-to-end   5.60 s  distinct=    47  valid=1.000  E=-479.151959 Ha


## 6. quimb `CircuitMPS`, on the CPU and the GPU
`qiskit-addon-aqc-tensor` converts the Qiskit circuit to quimb gates and applies them to a `CircuitMPS`. Two settings matter for a fair comparison:

- `cutoff=1e-8` alone is **not** a relative cutoff everywhere: `CircuitMPS` passes `cutoff_mode="rsum2"` (a cap on the discarded sum of squares) to some of its splits. `gate_opts={"cutoff_mode": "rel"}` makes it match cuQuantum's `rel_cutoff`.
- quimb is array-agnostic: passing `to_backend=cupy.asarray` keeps the gates, the MPS and the SVDs on the GPU.

quimb samples one shot at a time in Python, and each shot walks the MPS site by site. This loop, not the build, sets quimb's end-to-end time: about 4 ms a shot at 32 qubits, so 20,000 shots take about 80 s even on the CPU. On the GPU it is worse, since every step is a tiny kernel followed by a copy back to the CPU. So the GPU run builds the MPS on the GPU, copies it to the CPU, and samples there. The copy is counted in end-to-end. Samples are drawn from `psi.sample`, which yields each configuration with qubit 0 first (`CircuitMPS.sample` does the same and joins it into a string). cuQuantum draws all shots in one batch on the GPU, so its sampling takes well under a second.

Here quimb's build is about 5x faster on the GPU than on the CPU. With `MOLECULE = "n2"` the CPU wins, since 16-qubit tensors are too small to repay the GPU's overheads.

In [6]:
try:
    import quimb.tensor as qtn
    from functools import partial
    from qiskit_addon_aqc_tensor.simulation import tensornetwork_from_circuit
    from qiskit_addon_aqc_tensor.simulation.quimb import QuimbSimulator
    HAS_QUIMB = True
except ImportError as exc:
    HAS_QUIMB = False
    print("quimb skipped:", exc)

def quimb_run(gpu, shots=SHOTS):
    opts = {"to_backend": cupy.asarray} if gpu else {}
    settings = QuimbSimulator(partial(qtn.CircuitMPS, max_bond=CHI, cutoff=1e-8,
                                      gate_opts={"cutoff_mode": "rel"}, **opts))
    t = time.perf_counter()
    circ = tensornetwork_from_circuit(tqc, settings)
    if gpu:
        cupy.cuda.Device().synchronize()
    build = time.perf_counter() - t
    psi = circ.psi
    if gpu:
        psi.apply_to_arrays(cupy.asnumpy)                    # GPU -> CPU, then sample there
    bitstrings = Counter("".join(map(str, config)) for config, _ in psi.sample(shots, seed=1))
    counts = to_counts(bitstrings, qubit0_first=True)
    return counts, build, time.perf_counter() - t

if HAS_QUIMB:
    score("quimb (CPU)", *quimb_run(gpu=False))
    if HAS_GPU:
        quimb_run(gpu=True, shots=10)                        # warm-up: CuPy kernels compile
        score("quimb (GPU)", *quimb_run(gpu=True))

quimb (CPU)            build  53.03 s  end-to-end 134.53 s  distinct=    57  valid=1.000  E=-479.164545 Ha


quimb (GPU)            build   9.74 s  end-to-end  89.87 s  distinct=    57  valid=1.000  E=-479.164545 Ha


## 7. Qiskit Aer, `matrix_product_state`
Aer can sample the MPS directly: add measurements and run with `method="matrix_product_state"`. Its counts are Qiskit bitstrings, qubit 0 last. (`qiskit_addon_aqc_tensor.simulation.tensornetwork_from_circuit` with `QiskitAerSimulationSettings` returns the MPS itself instead.)

Build runs the circuit with `save_matrix_product_state` and no measurements. End-to-end is the run with measurements, up to the counts.

**Caution.** In our 32- and 38-qubit runs Aer's MPS came back wrong whenever the bond dimension cap actually truncated it: the norm rose above 1 and states at different caps had near-zero overlap with each other. Without truncation it was exact. Here the cap of 256 truncates, and it shows below: about 14% of Aer's shots break the particle number, which the exact state never does, and its build takes about 12 minutes. Do not rely on Aer's capped MPS at sizes that need truncation. (With `MOLECULE = "n2"` the cap never truncates and Aer is exact.)

In [7]:
try:
    from qiskit_aer import AerSimulator
    HAS_AER = True
except ImportError as exc:
    HAS_AER = False
    print("Aer skipped:", exc)

if HAS_AER:
    sim = AerSimulator(method="matrix_product_state", precision="double",
                       matrix_product_state_max_bond_dimension=CHI,
                       matrix_product_state_truncation_threshold=1e-16, seed_simulator=1)
    saved = tqc.copy()
    saved.save_matrix_product_state()
    t = time.perf_counter()
    sim.run(saved, shots=1).result()
    build = time.perf_counter() - t
    measured = tqc.copy()
    measured.measure_all()
    t = time.perf_counter()
    raw = sim.run(measured, shots=SHOTS).result().get_counts()
    score("Aer MPS", to_counts(raw, qubit0_first=False), build, time.perf_counter() - t)

Aer MPS                build 732.61 s  end-to-end 718.09 s  distinct=   565  valid=0.860  E=-479.181799 Ha


## 8. Compare
Each shot-based backend is compared with the mean of the exact shots and should land near their spread over seeds. The remaining differences are shot noise: each backend draws its own 20,000 shots, so the sampled subspaces differ. "sqd-gqe mps" and "cuQuantum (direct)" run the same simulation with the same settings and differ only in their shots. In the recorded run one lands inside the five-seed range and the other just outside it, so five seeds understate the noise. Aer is the exception: its energy is lower because its broken shots add determinants to the subspace, not because its state is closer to the exact one. The exact `topk` row reuses the ffsim build time, since it reads the same state vector. `mps_topk` draws no shots, so it is compared with the exact `topk` instead, and should match it. Its energy is higher than the shot-based ones here: the `1/shots` floor keeps only the determinants a 20,000-shot run expects to draw at least once, while the shots also catch some rarer ones by chance.

In [8]:
shots_ref, topk_ref = sum(spread) / len(spread), results["ffsim topk (exact)"]["energy"]
lo, hi = min(spread), max(spread)
print(f"exact shots: mean {shots_ref:.6f} Ha, seed spread {1e3 * (hi - lo):.1f} mHa")
print()
print(f"{'backend':22s} {'build (s)':>9s} {'e2e (s)':>8s} {'distinct':>9s} {'valid':>6s} "
      f"{'reference':>12s} {'E - ref (mHa)':>14s}  within spread")
for name, r in results.items():
    if "topk" in name:
        ref, label, inside = topk_ref, "exact topk", "-"
    else:
        ref, label = shots_ref, "exact shots"
        inside = "yes" if lo - 1e-9 <= r["energy"] <= hi + 1e-9 else "no"
    print(f"{name:22s} {fmt(r['build']):>9s} {fmt(r['e2e']):>8s} {r['distinct']:9d} "
          f"{r['valid']:6.3f} {label:>12s} {1e3 * (r['energy'] - ref):14.3f}  {inside}")

exact shots: mean -479.159089 Ha, seed spread 12.5 mHa

backend                build (s)  e2e (s)  distinct  valid    reference  E - ref (mHa)  within spread
ffsim (exact)               3.04     3.30        45  1.000  exact shots          2.494  yes
ffsim topk (exact)          3.04     3.26        18  1.000   exact topk          0.000  -
sqd-gqe mps                    -     5.62        55  1.000  exact shots         -3.359  yes
sqd-gqe mps_topk            4.46     5.19        18  1.000   exact topk          0.000  -
cuQuantum (direct)          5.09     5.60        47  1.000  exact shots          7.130  no
quimb (CPU)                53.03   134.53        57  1.000  exact shots         -5.457  yes
quimb (GPU)                 9.74    89.87        57  1.000  exact shots         -5.457  yes
Aer MPS                   732.61   718.09       565  0.860  exact shots        -22.710  no


## 9. Using an MPS backend in training
Training and inference reach MPS sampling through the config, and today only **cuQuantum** is wired in:

```bash
# shots from the MPS
sbatch --account=<project> slurm/train_mps.sh molecule=snoh3_m sampler.mps_max_bond=256
# the most probable determinants instead of shots
sbatch --account=<project> slurm/train_mps.sh molecule=snoh3_m sampler.kind=mps_topk \
    sqd.enlarge_method=None sqd.recover_iterations=0
```

quimb and Aer, as above, are for checking and comparing results. Measured on ROQUO (GB200) at 32 and 38 qubits with the production settings on every backend (complex64, a `1e-5` cut), cuQuantum built a converged MPS 1.2-1.5x faster than quimb on the GPU at the same fidelity, and 14-60x faster than Aer, whose truncated states were wrong. With complex128 and no cut, quimb on the CPU was about 10x slower than on the GPU.